# Pré-processamento e Modelagem - Risco de Crédito

Este notebook aplica o pré-processamento definido a partir da EDA e treina os cinco modelos de classificação (KNN, Árvore de Decisão, Random Forest, AdaBoost e MLP), comparando os resultados com o benchmark de Yang et al. (2025).


## 1. Importação e carregamento dos dados

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

# Carregar a tabela principal
df = pd.read_csv('../data/application_train.csv')

print(f"Dados carregados: {df.shape[0]} linhas e {df.shape[1]} colunas")

Dados carregados: 307511 linhas e 122 colunas


## 2. Tratamento da anomalia em DAYS_EMPLOYED

O valor 365.243 dias (~1000 anos) é um código placeholder presente em ~18% dos registros, indicando clientes sem vínculo empregatício. Substituímos por NaN e criamos uma flag binária, preservando essa informação como preditor.

In [4]:
# Criar flag que marca os registros com a anomalia (antes de substituir)
df['FLAG_SEM_EMPREGO'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)

# Substituir o valor anômalo por NaN
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

# Verificar o resultado
print("Flag criada - distribuição:")
print(df['FLAG_SEM_EMPREGO'].value_counts())
print(f"\nValores faltantes em DAYS_EMPLOYED agora: {df['DAYS_EMPLOYED'].isnull().sum()}")
print(f"Novo máximo de DAYS_EMPLOYED: {df['DAYS_EMPLOYED'].max():.0f} dias")

Flag criada - distribuição:
FLAG_SEM_EMPREGO
0    252137
1     55374
Name: count, dtype: int64

Valores faltantes em DAYS_EMPLOYED agora: 55374
Novo máximo de DAYS_EMPLOYED: 0 dias


## 3. Tratamento de valores faltantes

Estratégia definida a partir da EDA: descartar colunas com proporção excessiva de faltantes (> 60%) e imputar as demais: Mediana para variáveis numéricas (robusta a outliers) e moda para categóricas.

In [5]:
# Calcular o percentual de faltantes por coluna
pct_faltantes = (df.isnull().sum() / len(df)) * 100

# Identificar colunas com mais de 60% de faltantes
colunas_descartar = pct_faltantes[pct_faltantes > 60].index.tolist()

print(f"Colunas com mais de 60% de faltantes (serão descartadas): {len(colunas_descartar)}")
print(colunas_descartar)

df = df.drop(columns=colunas_descartar)

print(f"\nFormato após descarte: {df.shape[0]} linhas e {df.shape[1]} colunas")

Colunas com mais de 60% de faltantes (serão descartadas): 17
['OWN_CAR_AGE', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'FLOORSMIN_AVG', 'LIVINGAPARTMENTS_AVG', 'NONLIVINGAPARTMENTS_AVG', 'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'FLOORSMIN_MODE', 'LIVINGAPARTMENTS_MODE', 'NONLIVINGAPARTMENTS_MODE', 'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'FLOORSMIN_MEDI', 'LIVINGAPARTMENTS_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'FONDKAPREMONT_MODE']

Formato após descarte: 307511 linhas e 106 colunas


### 3.1 Separação de colunas numéricas e categóricas

As colunas são separadas por tipo, pois a imputação difere: numéricas recebem a mediana; categóricas, a moda.

In [6]:
# Separar colunas por tipo
colunas_numericas = df.select_dtypes(include=[np.number]).columns.tolist()
colunas_categoricas = df.select_dtypes(include=['object']).columns.tolist()

print(f"Colunas numéricas: {len(colunas_numericas)}")
print(f"Colunas categóricas: {len(colunas_categoricas)}")
print(f"\nExemplos de categóricas: {colunas_categoricas[:5]}")

Colunas numéricas: 91
Colunas categóricas: 15

Exemplos de categóricas: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE']


### 3.2 Imputação dos valores faltantes

Numéricas: preenchidas com a mediana (robusta a outliers). 

Categóricas: preenchidas com a moda (valor mais frequente).

In [7]:
# Imputar colunas NUMÉRICAS com a mediana
for col in colunas_numericas:
    if df[col].isnull().sum() > 0:
        mediana = df[col].median()
        df[col] = df[col].fillna(mediana)

# Imputar colunas CATEGÓRICAS com a moda (valor mais frequente)
for col in colunas_categoricas:
    if df[col].isnull().sum() > 0:
        moda = df[col].mode()[0]
        df[col] = df[col].fillna(moda)

# Verificar se ainda restam faltantes
total_faltantes = df.isnull().sum().sum()
print(f"Total de valores faltantes restantes: {total_faltantes}")

Total de valores faltantes restantes: 0


## 4. Codificação das variáveis categóricas

As variáveis de texto são convertidas em números. Estratégia: Label Encoding para binárias e One-Hot Encoding para as demais (categorias sem ordem natural).

In [8]:
# Investigar quantas categorias únicas cada coluna categórica tem
print("Número de categorias únicas por coluna categórica:\n")
for col in colunas_categoricas:
    n_unicas = df[col].nunique()
    print(f"{col}: {n_unicas} categorias")

Número de categorias únicas por coluna categórica:

NAME_CONTRACT_TYPE: 2 categorias
CODE_GENDER: 3 categorias
FLAG_OWN_CAR: 2 categorias
FLAG_OWN_REALTY: 2 categorias
NAME_TYPE_SUITE: 7 categorias
NAME_INCOME_TYPE: 8 categorias
NAME_EDUCATION_TYPE: 5 categorias
NAME_FAMILY_STATUS: 6 categorias
NAME_HOUSING_TYPE: 6 categorias
OCCUPATION_TYPE: 18 categorias
WEEKDAY_APPR_PROCESS_START: 7 categorias
ORGANIZATION_TYPE: 58 categorias
HOUSETYPE_MODE: 3 categorias
WALLSMATERIAL_MODE: 7 categorias
EMERGENCYSTATE_MODE: 2 categorias


In [9]:
# Investigar os valores da coluna CODE_GENDER
print("Valores em CODE_GENDER:")
print(df['CODE_GENDER'].value_counts())

Valores em CODE_GENDER:
CODE_GENDER
F      202448
M      105059
XNA         4
Name: count, dtype: int64


### 4.1 Limpeza do CODE_GENDER

A categoria "XNA" (não informado, apenas 4 registros) é substituída pela moda ("F"), tornando a variável binária.

In [10]:
# Substituir XNA (apenas 4 registros) pela moda
df['CODE_GENDER'] = df['CODE_GENDER'].replace('XNA', 'F')

# Confirmar que agora só há 2 categorias
print("CODE_GENDER após limpeza:")
print(df['CODE_GENDER'].value_counts())

CODE_GENDER após limpeza:
CODE_GENDER
F    202452
M    105059
Name: count, dtype: int64


### 4.2 Label Encoding das variáveis binárias

As variáveis com apenas 2 categorias são convertidas para 0 e 1.

In [11]:
from sklearn.preprocessing import LabelEncoder

# Colunas binárias (2 categorias cada)
colunas_binarias = ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR',
                    'FLAG_OWN_REALTY', 'EMERGENCYSTATE_MODE']

# Aplicar Label Encoding em cada uma
le = LabelEncoder()
for col in colunas_binarias:
    df[col] = le.fit_transform(df[col])
    print(f"{col}: {df[col].unique()}")

print("\nLabel Encoding aplicado nas binárias!")

NAME_CONTRACT_TYPE: [0 1]
CODE_GENDER: [1 0]
FLAG_OWN_CAR: [0 1]
FLAG_OWN_REALTY: [1 0]
EMERGENCYSTATE_MODE: [0 1]

Label Encoding aplicado nas binárias!


### 4.3 Label Encoding das variáveis com muitas categorias

`OCCUPATION_TYPE` (18) e `ORGANIZATION_TYPE` (58) recebem Label Encoding para evitar a explosão de colunas do One-Hot. Justifica-se pelo uso predominante de modelos baseados em árvores, robustos a esse tipo de codificação.

In [12]:
# Label Encoding nas colunas com muitas categorias
colunas_muitas_cat = ['OCCUPATION_TYPE', 'ORGANIZATION_TYPE']

for col in colunas_muitas_cat:
    df[col] = le.fit_transform(df[col])
    print(f"{col}: convertida para valores de 0 a {df[col].max()}")

print("\nLabel Encoding aplicado nas colunas grandes!")

OCCUPATION_TYPE: convertida para valores de 0 a 17
ORGANIZATION_TYPE: convertida para valores de 0 a 57

Label Encoding aplicado nas colunas grandes!


### 4.4 One-Hot Encoding das demais categóricas

As variáveis categóricas sem ordem natural e com poucas categorias são convertidas via One-Hot Encoding, criando uma coluna binária para cada categoria.

In [13]:
# Colunas que receberão One-Hot Encoding
colunas_onehot = ['NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
                  'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE',
                  'WEEKDAY_APPR_PROCESS_START', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE']

# Guardar o número de colunas antes
colunas_antes = df.shape[1]

# Aplicar One-Hot Encoding
df = pd.get_dummies(df, columns=colunas_onehot, drop_first=True)

# Converter colunas booleanas (True/False) para inteiros (1/0)
df = df.astype({col: 'int' for col in df.select_dtypes(include='bool').columns})

print(f"Colunas antes do One-Hot: {colunas_antes}")
print(f"Colunas depois do One-Hot: {df.shape[1]}")
print(f"Novas colunas criadas: {df.shape[1] - colunas_antes}")

Colunas antes do One-Hot: 106
Colunas depois do One-Hot: 139
Novas colunas criadas: 33


## 5. Separação de features (X) e alvo (y)

Os dados são divididos entre as variáveis preditoras (X) e a variável-alvo (y). O identificador `SK_ID_CURR` é removido por não ter valor preditivo.

In [14]:
# Separar features (X) e alvo (y)
# Remover TARGET (alvo) e SK_ID_CURR (apenas identificador)
X = df.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df['TARGET']

print(f"X (features): {X.shape[0]} linhas e {X.shape[1]} colunas")
print(f"y (alvo): {y.shape[0]} valores")
print(f"\nDistribuição do alvo:")
print(y.value_counts(normalize=True) * 100)

X (features): 307511 linhas e 137 colunas
y (alvo): 307511 valores

Distribuição do alvo:
TARGET
0    91.927118
1     8.072882
Name: proportion, dtype: float64


## 6. Divisão treino/teste (Holdout)

Os dados são divididos em 80% para treino e 20% para teste. Usa-se estratificação (`stratify`) para preservar a proporção de classes em ambos os conjuntos, e uma semente fixa (`random_state`) para reprodutibilidade.

In [15]:
from sklearn.model_selection import train_test_split

# Dividir em treino (80%) e teste (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,    # semente fixa: garante que a divisão seja sempre a mesma
    stratify=y          # mantém a proporção de classes (8% inadimplentes) nos dois
)

print(f"Treino: {X_train.shape[0]} linhas")
print(f"Teste:  {X_test.shape[0]} linhas")
print(f"\nProporção de inadimplentes no treino: {y_train.mean()*100:.2f}%")
print(f"Proporção de inadimplentes no teste:  {y_test.mean()*100:.2f}%")

Treino: 246008 linhas
Teste:  61503 linhas

Proporção de inadimplentes no treino: 8.07%
Proporção de inadimplentes no teste:  8.07%


## 7. Padronização das features

As features são padronizadas (média 0, desvio 1) com `StandardScaler`, essencial para modelos sensíveis a escala (KNN, MLP). O scaler é ajustado apenas no treino e aplicado a treino e teste, evitando vazamento de dados.

In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Ajustar (fit) APENAS no treino e transformar o treino
X_train_scaled = scaler.fit_transform(X_train)

# Apenas transformar o teste (usando os parâmetros aprendidos no treino)
X_test_scaled = scaler.transform(X_test)

print("Padronização concluída!")
print(f"Média das features no treino (deve ser ~0): {X_train_scaled.mean():.4f}")
print(f"Desvio padrão no treino (deve ser ~1): {X_train_scaled.std():.4f}")

Padronização concluída!
Média das features no treino (deve ser ~0): 0.0000
Desvio padrão no treino (deve ser ~1): 1.0000


## 8. Balanceamento com SMOTE

Para corrigir o desbalanceamento (8% de inadimplentes), aplica-se o SMOTE, que gera exemplos sintéticos da classe minoritária. **Aplicado apenas ao treino**, preservando a distribuição real no teste para uma avaliação fidedigna.

In [17]:
from imblearn.over_sampling import SMOTE
from collections import Counter

print(f"Antes do SMOTE - distribuição no treino: {Counter(y_train)}")

# Aplicar SMOTE APENAS no treino
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"Depois do SMOTE - distribuição no treino: {Counter(y_train_balanced)}")
print(f"\nTotal de amostras no treino após SMOTE: {len(y_train_balanced)}")

Antes do SMOTE - distribuição no treino: Counter({0: 226148, 1: 19860})
Depois do SMOTE - distribuição no treino: Counter({0: 226148, 1: 226148})

Total de amostras no treino após SMOTE: 452296


## 9. Modelagem

Treinamento e avaliação dos cinco modelos exigidos (KNN, Árvore de Decisão, Random Forest, AdaBoost e MLP), com as mesmas métricas do benchmark (Acurácia, Precisão, Recall, F1 e AUC) para comparação direta.

### 9.1 Função de avaliação

In [18]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

# Dicionário para guardar os resultados de todos os modelos
resultados = {}

def avaliar_modelo(nome, modelo, X_tr, y_tr, X_te, y_te):
    """Treina um modelo, faz previsões e calcula as 5 métricas."""
    # Treinar
    modelo.fit(X_tr, y_tr)

    # Prever no conjunto de teste
    y_pred = modelo.predict(X_te)
    y_proba = modelo.predict_proba(X_te)[:, 1]  # probabilidade da classe 1 (para AUC)

    # Calcular as 5 métricas
    metricas = {
        'Acuracia': accuracy_score(y_te, y_pred),
        'Precisao': precision_score(y_te, y_pred),
        'Recall': recall_score(y_te, y_pred),
        'F1': f1_score(y_te, y_pred),
        'AUC': roc_auc_score(y_te, y_proba)
    }

    # Guardar e exibir
    resultados[nome] = metricas
    print(f"=== {nome} ===")
    for m, v in metricas.items():
        print(f"{m}: {v:.4f}")
    print()

    return modelo

print("Função de avaliação criada!")

Função de avaliação criada!


### 9.2 Árvore de Decisão (baseline)

Primeiro modelo, usado como referência inicial. A Árvore de Decisão classifica através de uma sequência de divisões binárias nas features.

In [19]:
from sklearn.tree import DecisionTreeClassifier

# Criar e avaliar a Árvore de Decisão
arvore = DecisionTreeClassifier(random_state=42)
avaliar_modelo('Arvore de Decisao', arvore,
               X_train_balanced, y_train_balanced,
               X_test_scaled, y_test)

=== Arvore de Decisao ===
Acuracia: 0.8289
Precisao: 0.1254
Recall: 0.1875
F1: 0.1503
AUC: 0.5364



,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

### 9.3 Random Forest

Conjunto (ensemble) de múltiplas árvores de decisão que votam pela classificação final. Mais robusto e preciso que uma árvore isolada, por reduzir o overfitting.

In [20]:
from sklearn.ensemble import RandomForestClassifier

# Criar e avaliar o Random Forest
# n_jobs=-1 usa todos os núcleos do processador (acelera o treino)
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
avaliar_modelo('Random Forest', rf,
               X_train_balanced, y_train_balanced,
               X_test_scaled, y_test)

=== Random Forest ===
Acuracia: 0.9156
Precisao: 0.2684
Recall: 0.0264
F1: 0.0480
AUC: 0.7033



,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total

### 9.4 AdaBoost

Modelo de boosting que treina árvores sequencialmente, cada uma corrigindo os erros da anterior, focando progressivamente nos casos mais difíceis.

In [21]:
from sklearn.ensemble import AdaBoostClassifier

# Criar e avaliar o AdaBoost
ada = AdaBoostClassifier(n_estimators=100, random_state=42)
avaliar_modelo('AdaBoost', ada,
               X_train_balanced, y_train_balanced,
               X_test_scaled, y_test)

=== AdaBoost ===
Acuracia: 0.8481
Precisao: 0.1796
Recall: 0.2471
F1: 0.2080
AUC: 0.6890



,"n_estimators n_estimators: int, default=50The maximum number of estimators at which boosting is terminated.In case of perfect fit, the learning procedure is stopped early.Values must be in the range `[1, inf)`.",100
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given at each `estimator` at eachboosting iteration.Thus, it is only used when `estimator` exposes a `random_state`.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"estimator estimator: object, default=NoneThe base estimator from which the boosted ensemble is built.Support for sample weighting is required, as well as proper``classes_`` and ``n_classes_`` attributes. If ``None``, thenthe base estimator is :class:`~sklearn.tree.DecisionTreeClassifier`initialized with `max_depth=1`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",None
,"learning_rate learning_rate: float, default=1.0Weight applied to each classifier at each boosting iteration. A higherlearning rate increases the contribution of each classifier. There isa trade-off between the `learning_rate` and `n_estimators` parameters.Values must be in the range `(0.0, inf)`.",1.0
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels.","ndarray[int64](2,)","[0,1]"
estimator_ estimator_: estimatorThe base estimator from which the ensemble is grown... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,DecisionTreeClassifier,DecisionTreeC...r(max_depth=1)
estimator_errors_ estimator_errors_: ndarray of floatsClassification error for each estimator in the boostedensemble.,"ndarray[float64](100,)","[0.36,0.37,0.44,...,0.46,0.46,0.46]"
estimator_weights_ estimator_weights_: ndarray of floatsWeights for each estimator in the boosted ensemble.,"ndarray[float64](100,)","[0.59,0.52,0.24,...,0.16,0.18,0.16]"
estimators_ estimators_: list of classifiersThe collection of fitted sub-estimators.,list,"[DecisionTreeC...te=1608637542), DecisionTreeC...te=1273642419), DecisionTreeC...te=1935803228), DecisionTreeC...ate=787846414), ...]"
"feature_importances_ feature_importances_: ndarray of shape (n_features,)The impurity-based feature importances if supported by the``estimator`` (when based on decision trees).Warning: impurity-based feature importances can be misleading forhigh cardinality features (many unique values). See:func:`sklearn.inspection.permutation_importance` as an alternative.","ndarray[float64](137,)","[0. ,0.04,0.08,...,0. ,0. ,0. ]"


### 9.5 KNN (K-Nearest Neighbors)

Classifica cada cliente com base nos K vizinhos mais próximos. Por ser computacionalmente intensivo na predição, utiliza-se uma amostra do conjunto de treino balanceado como base de comparação, mantendo a qualidade com viabilidade computacional.

In [22]:
from sklearn.neighbors import KNeighborsClassifier

# Reduzir o treino para uma amostra (KNN é lento com muitos dados)
# Usar 50.000 amostras aleatórias do treino balanceado
np.random.seed(42)
indices_amostra = np.random.choice(len(X_train_balanced), size=50000, replace=False)
X_train_knn = X_train_balanced[indices_amostra]
y_train_knn = y_train_balanced.iloc[indices_amostra] if hasattr(y_train_balanced, 'iloc') else y_train_balanced[indices_amostra]

# Criar e avaliar o KNN
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
avaliar_modelo('KNN', knn,
               X_train_knn, y_train_knn,
               X_test_scaled, y_test)

=== KNN ===
Acuracia: 0.3898
Precisao: 0.0957
Recall: 0.7764
F1: 0.1704
AUC: 0.5977



,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",-1
,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


### 9.6 MLP (Multi-Layer Perceptron)

Rede neural feedforward com camadas ocultas, capaz de capturar relações não-lineares complexas. Utiliza-se early stopping para interromper o treino quando o desempenho estabiliza, otimizando o tempo de treinamento.

In [23]:
from sklearn.neural_network import MLPClassifier

# Criar e avaliar o MLP
# hidden_layer_sizes: 2 camadas ocultas (100 e 50 neurônios)
# early_stopping: para o treino quando não melhora mais (economiza tempo)
# max_iter: limite de iterações
mlp = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    max_iter=100,
    early_stopping=True,
    random_state=42
)
avaliar_modelo('MLP', mlp,
               X_train_balanced, y_train_balanced,
               X_test_scaled, y_test)

=== MLP ===
Acuracia: 0.8396
Precisao: 0.1722
Recall: 0.2594
F1: 0.2070
AUC: 0.6567



,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(100, ...)"
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",100
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"early_stopping early_stopping: bool, default=FalseWhether to use early stopping to terminate training when validationscore is not improving. If set to True, it will automatically setaside ``validation_fraction`` of training data as validation andterminate training when validation score is not improving by at least``tol`` for ``n_iter_no_change`` consecutive epochs. The split isstratified, except in a multilabel setting.If early stopping is False, then the training stops when the trainingloss does not improve by more than ``tol`` for ``n_iter_no_change``consecutive passes over the training set.Only effective when solver='sgd' or 'adam'.",True
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating

## 10. Otimização de Hiperparâmetros (Grid Search + Validação Cruzada)

Otimização dos cinco modelos via Grid Search com validação cruzada (3 folds), usando AUC como métrica de seleção. As grades foram definidas de forma enxuta, priorizando os hiperparâmetros mais relevantes e mantendo viabilidade computacional.

### 10.1 Árvore de Decisão otimizada

In [24]:
from sklearn.model_selection import GridSearchCV

# Grade de hiperparâmetros para a Árvore de Decisão
param_arvore = {
    'max_depth': [5, 10, 20],
    'min_samples_split': [2, 10],
    'criterion': ['gini', 'entropy']
}

# Grid Search com validação cruzada (3 folds), otimizando AUC
grid_arvore = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_arvore,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1
)

grid_arvore.fit(X_train_balanced, y_train_balanced)

print(f"Melhores parâmetros: {grid_arvore.best_params_}")
print(f"Melhor AUC (validação cruzada): {grid_arvore.best_score_:.4f}")

# Avaliar o melhor modelo no teste
avaliar_modelo('Arvore Otimizada', grid_arvore.best_estimator_,
               X_train_balanced, y_train_balanced,
               X_test_scaled, y_test)

Melhores parâmetros: {'criterion': 'entropy', 'max_depth': 20, 'min_samples_split': 10}
Melhor AUC (validação cruzada): 0.9255
=== Arvore Otimizada ===
Acuracia: 0.8463
Precisao: 0.1456
Recall: 0.1857
F1: 0.1632
AUC: 0.6099



,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'entropy'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

### 10.2 Random Forest otimizado

In [26]:
# Grade REDUZIDA para o Random Forest (otimizada para tempo)
param_rf = {
    'n_estimators': [100],
    'max_depth': [10, 20]
}

# Grid Search com validação cruzada (2 folds para acelerar)
grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_rf,
    cv=2,
    scoring='roc_auc',
    n_jobs=-1
)

grid_rf.fit(X_train_balanced, y_train_balanced)

print(f"Melhores parâmetros: {grid_rf.best_params_}")
print(f"Melhor AUC (validação cruzada): {grid_rf.best_score_:.4f}")

avaliar_modelo('Random Forest Otimizado', grid_rf.best_estimator_,
               X_train_balanced, y_train_balanced,
               X_test_scaled, y_test)

Melhores parâmetros: {'max_depth': 20, 'n_estimators': 100}
Melhor AUC (validação cruzada): 0.9773
=== Random Forest Otimizado ===
Acuracia: 0.8961
Precisao: 0.2243
Recall: 0.1170
F1: 0.1538
AUC: 0.6975



,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total n

### 10.3 AdaBoost otimizado

In [30]:
# AdaBoost com configuração ajustada (sem Grid Search, por custo computacional)
ada_otimizado = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=0.5,
    random_state=42
)

avaliar_modelo('AdaBoost Otimizado', ada_otimizado,
               X_train_balanced, y_train_balanced,
               X_test_scaled, y_test)

=== AdaBoost Otimizado ===
Acuracia: 0.7799
Precisao: 0.1644
Recall: 0.4228
F1: 0.2367
AUC: 0.6920



,"n_estimators n_estimators: int, default=50The maximum number of estimators at which boosting is terminated.In case of perfect fit, the learning procedure is stopped early.Values must be in the range `[1, inf)`.",100
,"learning_rate learning_rate: float, default=1.0Weight applied to each classifier at each boosting iteration. A higherlearning rate increases the contribution of each classifier. There isa trade-off between the `learning_rate` and `n_estimators` parameters.Values must be in the range `(0.0, inf)`.",0.5
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given at each `estimator` at eachboosting iteration.Thus, it is only used when `estimator` exposes a `random_state`.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"estimator estimator: object, default=NoneThe base estimator from which the boosted ensemble is built.Support for sample weighting is required, as well as proper``classes_`` and ``n_classes_`` attributes. If ``None``, thenthe base estimator is :class:`~sklearn.tree.DecisionTreeClassifier`initialized with `max_depth=1`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",None
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels.","ndarray[int64](2,)","[0,1]"
estimator_ estimator_: estimatorThe base estimator from which the ensemble is grown... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,DecisionTreeClassifier,DecisionTreeC...r(max_depth=1)
estimator_errors_ estimator_errors_: ndarray of floatsClassification error for each estimator in the boostedensemble.,"ndarray[float64](100,)","[0.36,0.37,0.43,...,0.47,0.48,0.47]"
estimator_weights_ estimator_weights_: ndarray of floatsWeights for each estimator in the boosted ensemble.,"ndarray[float64](100,)","[0.29,0.26,0.13,...,0.06,0.05,0.06]"
estimators_ estimators_: list of classifiersThe collection of fitted sub-estimators.,list,"[DecisionTreeC...te=1608637542), DecisionTreeC...te=1273642419), DecisionTreeC...te=1935803228), DecisionTreeC...ate=787846414), ...]"
"feature_importances_ feature_importances_: ndarray of shape (n_features,)The impurity-based feature importances if supported by the``estimator`` (when based on decision trees).Warning: impurity-based feature importances can be misleading forhigh cardinality features (many unique values). See:func:`sklearn.inspection.permutation_importance` as an alternative.","ndarray[float64](137,)","[0. ,0.05,0.05,...,0. ,0. ,0. ]"


### 10.4 KNN otimizado

In [31]:
# KNN com configuração ajustada (testando mais vizinhos para suavizar)
knn_otimizado = KNeighborsClassifier(
    n_neighbors=15,      # mais vizinhos = decisão mais estável
    weights='distance',  # vizinhos mais próximos pesam mais
    n_jobs=-1
)

avaliar_modelo('KNN Otimizado', knn_otimizado,
               X_train_knn, y_train_knn,
               X_test_scaled, y_test)

=== KNN Otimizado ===
Acuracia: 0.3119
Precisao: 0.0943
Recall: 0.8743
F1: 0.1702
AUC: 0.6364



,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",15
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'distance'
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",-1
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


### 10.5 MLP otimizado

In [32]:
# MLP com configuração ajustada
mlp_otimizado = MLPClassifier(
    hidden_layer_sizes=(128, 64),  # arquitetura um pouco maior
    activation='relu',
    alpha=0.001,                   # regularização para evitar overfitting
    max_iter=150,
    early_stopping=True,
    random_state=42
)

avaliar_modelo('MLP Otimizado', mlp_otimizado,
               X_train_balanced, y_train_balanced,
               X_test_scaled, y_test)

=== MLP Otimizado ===
Acuracia: 0.8318
Precisao: 0.1486
Recall: 0.2290
F1: 0.1802
AUC: 0.6390



,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(128, ...)"
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.001
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",150
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"early_stopping early_stopping: bool, default=FalseWhether to use early stopping to terminate training when validationscore is not improving. If set to True, it will automatically setaside ``validation_fraction`` of training data as validation andterminate training when validation score is not improving by at least``tol`` for ``n_iter_no_change`` consecutive epochs. The split isstratified, except in a multilabel setting.If early stopping is False, then the training stops when the trainingloss does not improve by more than ``tol`` for ``n_iter_no_change``consecutive passes over the training set.Only effective when solver='sgd' or 'adam'.",True
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating 

## 11. Rastreamento de Experimentos com MLflow

Registro de todos os modelos no MLflow: parâmetros, métricas e o próprio modelo salvo. Cada modelo é uma run dentro do experimento, permitindo comparação organizada.

In [34]:
import mlflow
import mlflow.sklearn

# Usar SQLite como backend (versões recentes do MLflow pedem banco de dados)
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# Criar/definir o experimento
mlflow.set_experiment("Credit Risk - Default Prediction")

print("MLflow configurado!")
print("Backend: SQLite (mlflow.db)")
print("Experimento: Credit Risk - Default Prediction")

2026/06/15 15:24:38 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/15 15:24:39 INFO mlflow.store.db.utils: Updating database tables
2026/06/15 15:24:40 INFO mlflow.tracking.fluent: Experiment with name 'Credit Risk - Default Prediction' does not exist. Creating a new experiment.


MLflow configurado!
Backend: SQLite (mlflow.db)
Experimento: Credit Risk - Default Prediction


### 11.1 Registro dos modelos no MLflow

Cada modelo é registrado como uma run, com seus parâmetros, as cinco métricas e o modelo serializado.

In [35]:
# Função para registrar um modelo no MLflow
def registrar_mlflow(nome, modelo, metricas):
    """Registra um modelo no MLflow: parâmetros, métricas e o modelo serializado."""
    with mlflow.start_run(run_name=nome):
        # Registrar os parâmetros do modelo
        params = modelo.get_params()
        for p, v in params.items():
            mlflow.log_param(p, v)

        # Registrar as 5 métricas
        for metrica, valor in metricas.items():
            mlflow.log_metric(metrica, valor)

        # Salvar o modelo treinado
        mlflow.sklearn.log_model(modelo, name="modelo")

    print(f"✓ {nome} registrado no MLflow")

# Registrar todos os modelos otimizados
registrar_mlflow('Arvore Otimizada', grid_arvore.best_estimator_, resultados['Arvore Otimizada'])
registrar_mlflow('Random Forest Otimizado', grid_rf.best_estimator_, resultados['Random Forest Otimizado'])
registrar_mlflow('AdaBoost Otimizado', ada_otimizado, resultados['AdaBoost Otimizado'])
registrar_mlflow('KNN Otimizado', knn_otimizado, resultados['KNN Otimizado'])
registrar_mlflow('MLP Otimizado', mlp_otimizado, resultados['MLP Otimizado'])

print("\nTodos os modelos foram registrados no MLflow!")

2026/06/15 18:23:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✓ Arvore Otimizada registrado no MLflow


2026/06/15 18:24:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✓ Random Forest Otimizado registrado no MLflow


2026/06/15 18:24:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✓ AdaBoost Otimizado registrado no MLflow


2026/06/15 18:25:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✓ KNN Otimizado registrado no MLflow


2026/06/15 18:25:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✓ MLP Otimizado registrado no MLflow

Todos os modelos foram registrados no MLflow!
